In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import math
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from backtesting import Strategy, Backtest
from datetime import datetime, timedelta
from scipy.stats import linregress
import warnings
warnings.filterwarnings("ignore")
import plotly.io as pio
pio.renderers.default = "notebook_connected"

/Users/benlockhart/Downloads/BenFlix/.venv/lib/python3.14/site-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [2]:
url = "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv"

sp500 = pd.read_csv(url)

tickers = sp500["Symbol"].tolist()
tickers = [t.replace(".", "-") for t in tickers]

print(len(tickers))
print(tickers[:10])

503
['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A']


In [3]:
data = yf.download(
    tickers,
    period="10y",
    interval="1d",
    group_by="ticker",
    progress=False,
)

close = data.xs("Close", axis=1, level=1)
ma200 = close.rolling(200).mean()
valid = close.notna() & ma200.notna()
above = ((close > ma200) & valid).sum(axis=1)
below = ((close < ma200) & valid).sum(axis=1)

net = above / (above + below)

In [4]:
spx_df = yf.download('^GSPC', period="10y", interval="1d")
spx_df = spx_df.reset_index()
spx_df.columns = spx_df.columns.get_level_values(0)

merged_data = pd.concat([above, below, net], axis=1)
merged_data.columns = ['Above', 'Below', 'Net']
merged_data.index = pd.to_datetime(merged_data.index)

df = pd.merge(spx_df, merged_data, left_on='Date', right_index=True, how='left')
df['Date'] = pd.to_datetime(df['Date'])
df.dropna(inplace=True)
df

[*********************100%***********************]  1 of 1 completed


,Date,Close,High,Low,Open,Volume,Above,Below,Net
199,2017-08-15,2464.610107,2468.899902,2461.610107,2468.659912,2922430000,369,99,0.788462
200,2017-08-16,2468.110107,2474.929932,2463.860107,2468.629883,2956570000,368,100,0.786325
201,2017-08-17,2430.010010,2465.020020,2430.010010,2462.949951,3164950000,340,129,0.724947
202,2017-08-18,2425.550049,2440.270020,2420.689941,2427.639893,3426970000,335,134,0.714286
203,2017-08-21,2428.370117,2430.580078,2417.350098,2425.500000,2803650000,332,137,0.707889
...,...,...,...,...,...,...,...,...,...
2507,2026-10-21,7757.169922,7770.399902,7709.339844,7736.430176,4921480000,282,218,0.564000
2508,2026-10-22,7663.430176,7702.939941,7637.359863,7694.620117,5972620000,300,200,0.600000
2509,2026-10-23,7623.680176,7660.149902,7612.069824,7637.180176,5062330000,293,207,0.586000
2510,2026-10-26,7676.770020,7702.129883,7639.700195,7639.700195,5838070000,303,197,0.606000


In [5]:
def Full_STO(data, n1, n2, n3):
    hi = data.Net.rolling(n1).max()
    lo = data.Net.rolling(n1).min()
    k = ((data.Net - lo) / (hi - lo)) * 100
    kx = k.rolling(n3).mean()
    d = kx.rolling(n2).mean()
    return kx, d

df["Full_K"], df["Full_D"] = Full_STO(df, 14, 3, 3)
df

,Date,Close,High,Low,Open,Volume,Above,Below,Net,Full_K,Full_D
199,2017-08-15,2464.610107,2468.899902,2461.610107,2468.659912,2922430000,369,99,0.788462,NaN,NaN
200,2017-08-16,2468.110107,2474.929932,2463.860107,2468.629883,2956570000,368,100,0.786325,NaN,NaN
201,2017-08-17,2430.010010,2465.020020,2430.010010,2462.949951,3164950000,340,129,0.724947,NaN,NaN
202,2017-08-18,2425.550049,2440.270020,2420.689941,2427.639893,3426970000,335,134,0.714286,NaN,NaN
203,2017-08-21,2428.370117,2430.580078,2417.350098,2425.500000,2803650000,332,137,0.707889,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2507,2026-10-21,7757.169922,7770.399902,7709.339844,7736.430176,4921480000,282,218,0.564000,91.534392,91.534392
2508,2026-10-22,7663.430176,7702.939941,7637.359863,7694.620117,5972620000,300,200,0.600000,100.000000,94.356261
2509,2026-10-23,7623.680176,7660.149902,7612.069824,7637.180176,5062330000,293,207,0.586000,96.410256,95.981549
2510,2026-10-26,7676.770020,7702.129883,7639.700195,7639.700195,5838070000,303,197,0.606000,96.410256,97.606838


In [6]:
def signal(data):
    signal = [0] * len(df)
    for i in range(0,len(df)):
        if (df.Net.iloc[i-1] < 0.5) and (df.Net.iloc[i] > 0.5):
            signal[i] = 1
        elif (df.Full_K.iloc[i-1] < df.Full_D.iloc[i-1]) and (df.Full_K.iloc[i] > df.Full_D.iloc[i]):
            signal[i] = 2
        else:
            signal[i] = 0
        df["signal"] = signal
        
signal(df)
df

def oversold_entries(x):
    offset = 0.002
    if x['signal']==1:
        return x['Low'] * (1-offset)
    else:
        return np.nan

def long_entries(x):
    offset = 0.002
    if x['signal']==2:
        return x['Low'] * (1-offset)
    else:
        return np.nan

df['oversold_entries'] = df.apply(lambda x: oversold_entries(x), axis=1)
df['long_entries'] = df.apply(lambda x: long_entries(x), axis=1)
print(df['signal'].value_counts())
df.shape

signal
0    2031
2     252
1      30
Name: count, dtype: int64


(2313, 14)

In [7]:
df.set_index('Date', inplace=True)

In [12]:
bar = 2000
df1 = df[bar:bar+350].copy()

fig = make_subplots(rows=3, cols=1, 
                    shared_xaxes=True, 
                    row_heights=[0.50, 0.25, 0.25], 
                    vertical_spacing=0.05)

fig.add_trace(go.Candlestick(x=df1.index,
                open=df1['Open'],
                high=df1['High'],
                low=df1['Low'],
                close=df1['Close'],
                increasing_line_color = 'rgba(19,156,19,0.8)',
                decreasing_line_color = 'rgba(175,07,49,0.8)',
                name = 'SPX'),
                 row=1, col=1)

fig.add_scatter(x=df1.index, y=df1['oversold_entries'], mode="markers",
                marker=dict(size=9, symbol='arrow-up', color="White"),
                name="Oversold")

fig.add_scatter(x=df1.index, y=df1['long_entries'], mode="markers",
                marker=dict(size=7, color="gold"),
                name="Momentum")

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.Net, 
                         name="BP Index",
                         showlegend=False,
                        opacity=0.8,
                        line=dict(color='#00df9a', width=2)),
                         row=2, col=1)

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df.Full_K, 
                         line=dict(color='red', width=1),
                         name='Full%K',
                         showlegend=False),
                         row=3, col=1)

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df.Full_D, 
                         line=dict(color='white', width=1),
                         name='Full%D',
                         showlegend=False),
                         row=3, col=1)

fig.update_layout(
    annotations=[dict(text="BULL % Index",
                    font=dict(color="white", size=12),
                    xref="paper",
                    yref="paper",
                    x=0.03,
                    y=0.5,
                    showarrow=False),
                 dict(text="SPX Price Chart",
                    font=dict(color="white", size=12),
                    xref="paper",
                    yref="paper",
                    x=0.03,
                    y=1.03,
                    showarrow=False),
                dict(text="BULL % Stochastics",
                    font=dict(color="white", size=12),
                    xref="paper",
                    yref="paper",
                    x=0.03,
                    y=0.24,
                    showarrow=False)])

fig.add_hline(y=0.5, line_width=0.5, line_color="grey", row=2, col=1)
fig.add_hline(y=20, line_width=0.5, line_color="grey", row=3, col=1)
fig.add_hline(y=80, line_width=0.5, line_color="grey", row=3, col=1)


fig.update_layout(autosize=False, width=1100, height=900, 
                  xaxis_rangeslider_visible=False, 
                  template="plotly_dark")

fig.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"]),
    ])

fig.update_xaxes(gridcolor="#171717") 
fig.update_yaxes(gridcolor="#171717") 

fig.show()

In [9]:

def SIGNAL():
    return df.signal

class MyStrat(Strategy):
    
    def init(self):
        super().init()
        self.signal = self.I(SIGNAL)
    
    def next(self):
        super().next()
        
        price = self.data.Close[-1]
        
        if self.signal==1: 
            if self.position.is_short or not self.position:
                self.position.close()
                self.buy(size=0.99, tp=1.02*price)

        elif self.signal==2: 
            if self.position.is_short or not self.position:
                self.position.close()
                self.buy(size=0.99, tp=1.03*price)

        
bt = Backtest(df, MyStrat, cash=100_000, margin=1, exclusive_orders=True, commission=0.0005)
stats = bt.run()
stats

Backtest.run:   0%|          | 0/2312 [00:00<?, ?bar/s]

Start                     2017-08-15 00:00:00
End                       2026-10-27 00:00:00
Duration                   3360 days 00:00:00
Exposure Time [%]                    85.17077
Equity Final [$]                 321285.36787
Equity Peak [$]                  325234.90107
Commissions [$]                    7854.89481
Return [%]                          221.28537
Buy & Hold Return [%]               213.33963
Return (Ann.) [%]                    13.56002
Volatility (Ann.) [%]                20.25422
CAGR [%]                              9.14826
Sharpe Ratio                          0.66949
Sortino Ratio                         1.08223
Calmar Ratio                          0.40558
Alpha [%]                            25.62533
Beta                                  0.91713
Max. Drawdown [%]                   -33.43401
Avg. Drawdown [%]                    -1.87764
Max. Drawdown Duration      623 days 00:00:00
Avg. Drawdown Duration       18 days 00:00:00
# Trades                          